In [ ]:
import numpy as np
import rerun as rr
import rerun.blueprint as rrb

[Rotation representations](/aiml-common/lectures/motion-algebras/rotation-representations/index) described a body's orientation. [Angular velocity](/aiml-common/lectures/motion-algebras/angular-velocity/index) described how it changes, and [exponential coordinates](/aiml-common/lectures/motion-algebras/exponential-coordinates/index) showed how a finite rotation follows from it. This section adds position. A pose pairs a rotation with a translation. A twist pairs an angular velocity with a linear velocity. A screw motion is the exponential of a twist. In [Lie I](/aiml-common/lectures/motion-algebras/lie-theory/index), these are the group $SE(3)$, its Lie algebra $\mathfrak{se}(3)$, and the exponential between them.

The section starts in the plane. There every object is a $3 \times 3$ matrix you can compute by hand, and a TurtleBot serves as the running example. It then takes the same steps in three dimensions. The presentation follows sections 3.1 and 3.3 of Lynch and Park [@Kevin-M-Lynch2017-mp].

## Motion in the plane

### Poses in SE(2)

A pose in the plane is a position $(x, y)$ and a heading $\theta$. Moving a point $\mathbf{r}$ by a pose takes two steps: rotate it by $R(\theta)$, then shift it by $\mathbf{p} = (x, y)$. The result is $R(\theta)\,\mathbf{r} + \mathbf{p}$. The rotation is a matrix product, but the shift is an addition, so the two steps do not combine into one matrix.

[Homogeneous coordinates](/aiml-common/lectures/motion-algebras/homogeneous-coordinates/index) fix this with one extra number. Write the point $\mathbf{r} = (r_1, r_2)$ as $(r_1, r_2, 1)$. A $3 \times 3$ matrix can then do both steps in one product:

$$
\begin{bmatrix} R(\theta) & \mathbf{p} \\ 0 & 1 \end{bmatrix} \begin{bmatrix} \mathbf{r} \\ 1 \end{bmatrix} = \begin{bmatrix} R(\theta)\,\mathbf{r} + \mathbf{p} \\ 1 \end{bmatrix} .
$$

The 1 in the last row of the point multiplies $\mathbf{p}$, which adds the shift. This matrix is the *homogeneous transformation matrix* of the pose:

$$
T = \begin{bmatrix} R(\theta) & \mathbf{p} \\ 0 & 1 \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta & x \\ \sin\theta & \cos\theta & y \\ 0 & 0 & 1 \end{bmatrix} \in SE(2) .
$$

The configuration is the triple $(\theta, x, y)$. For a differential-drive robot such as the TurtleBot, $(x, y)$ is the position of the chassis and $\theta$ is its heading. The headings $\theta$ and $\theta + 2\pi$ are identical. The matrix $R(\theta)$ converts coordinates between the robot and world frames. A velocity $v_b = (v, 0)$ along the robot's heading becomes $v_s = R(\theta)\,v_b = (v\cos\theta, v\sin\theta)$ in the world frame.

The last coordinate tells the matrix what kind of object it is moving. A point, such as the position of a wheel, has a 1 there, so it is rotated and shifted. A direction, such as a velocity, has a 0 there. The 0 multiplies $\mathbf{p}$, so the direction is only rotated. This matches what happens to the robot: driving it somewhere else moves the points on it, but a direction has no position to move, so it only turns with the robot. For example, take $\theta = 90^\circ$ and $\mathbf{p} = (1, 2)$. The point $(1, 0, 1)$ goes to $(1, 3, 1)$, while the direction $(1, 0, 0)$ goes to $(0, 1, 0)$.

The inverse of a homogeneous transformation matrix is

$$
T^{-1} = \begin{bmatrix} R^T & -R^T \mathbf{p} \\ 0 & 1 \end{bmatrix} ,
$$

and transformations compose by matrix multiplication, $T_{ac} = T_{ab} T_{bc}$. Composing two poses shows the interaction between translation and rotation:

$$
T_1 T_2 =
\begin{bmatrix} R_1 & p_1 \\ 0 & 1 \end{bmatrix}
\begin{bmatrix} R_2 & p_2 \\ 0 & 1 \end{bmatrix}
=
\begin{bmatrix} R_1 R_2 & p_1 + R_1 p_2 \\ 0 & 1 \end{bmatrix} .
$$

The headings combine by multiplying their rotations, which adds the angles. The first rotation acts on the second translation, producing the term $R_1 p_2$. As a manifold, SE(2) looks like $\mathbb{R}^2 \times SO(2)$, a position paired with a heading. Its group structure also includes the action of rotation on translation. This structure is called a *semidirect product* and is written $SE(2) = \mathbb{R}^2 \rtimes SO(2)$.

### Twists in the plane

A planar body turns at a rate $\omega = \dot\theta$, and its origin moves with a linear velocity $(v_x, v_y)$. A *twist* collects the three numbers in one vector, $\mathcal{V} = (\omega, v_x, v_y) \in \mathbb{R}^3$. This order follows Lynch and Park: turning rate first, then the linear velocity. The matrix form of the twist belongs to $\mathfrak{se}(2)$, the Lie algebra of $SE(2)$:

$$
[\mathcal{V}] = \begin{bmatrix} [\omega] & \mathbf{v} \\ 0 & 0 \end{bmatrix} = \begin{bmatrix} 0 & -\omega & v_x \\ \omega & 0 & v_y \\ 0 & 0 & 0 \end{bmatrix} \in \mathfrak{se}(2) .
$$

As with [angular velocity](/aiml-common/lectures/motion-algebras/angular-velocity/index), you can express a twist in two frames. The *body twist* comes from $[\mathcal{V}_b] = T^{-1}\dot{T}$. The *space twist* comes from $[\mathcal{V}_s] = \dot{T} T^{-1}$. Both have the same turning rate, $\omega_b = \omega_s = \dot\theta$, because every rotation in the plane is about the same axis, out of the page. The linear parts differ:

- $\mathbf{v}_b = R^\top \dot{\mathbf{p}}$ is the velocity of the body-frame origin, written in the coordinates of $\{b\}$.

- $\mathbf{v}_s = \dot{\mathbf{p}} - [\omega]\mathbf{p} = (\dot x + \dot\theta\, y,\ \dot y - \dot\theta\, x)$ is the velocity of the point of the body currently at the origin of $\{s\}$, written in $\{s\}$. That point may lie outside the physical body. You can imagine extending the body far enough to contain it.

The body twist is related to the rate of change of the configuration $q = (\theta, x, y)$ by

$$
\mathcal{V}_b = \begin{bmatrix} \omega_b \\ v_{bx} \\ v_{by} \end{bmatrix} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos\theta & \sin\theta \\ 0 & -\sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} \dot\theta \\ \dot x \\ \dot y \end{bmatrix} .
$$

### Planar screw motion

Any planar displacement that includes a rotation is a rotation about a fixed point $\mathbf{s}$. A pure translation is the exception, because its fixed point has moved off to infinity. The figure shows such a motion.

![Planar screw motion.](images/2d-screw-motion.png)

(a) The body is rotated by $R$ about the origin of $\{s\}$ and then translated by $\mathbf{p}$. [Displacing a frame](/aiml-common/lectures/motion-algebras/rotation-representations/index#displacing-a-frame) works through this motion step by step.

(b) The rotation and translation can occur simultaneously. The displacement is then a rotation of $\beta = 90^\circ$ about a fixed point $\mathbf{s}$.

<video
  src="https://artifacts.aegeanai.com/manim/planar-motion/PlanarScrewMotion.mp4?v=4"
  controls
  loop
  muted
  playsInline
  width="100%"
  style={{ borderRadius: 8 }}
/>

*Two motions produce the same displacement. In the first, the body rotates by $R$ about the origin of $\{s\}$ and then translates by $\mathbf{p}$. In the second, it rotates by $\beta = 90^\circ$ about $\mathbf{s} = (0, 2)$. Both motions end at the same pose.*

The three screw coordinates $(\beta, s_x, s_y)$ parametrize the displacement. Here, $(s_x, s_y) = (0, 2)$ gives the coordinates of the point $\mathbf{s}$ in the fixed frame $\{s\}$. This point is the screw axis out of the page. The next part shows that a body moving with a constant twist makes exactly this kind of motion.

### From velocity to motion

For rotations, [angular velocity](/aiml-common/lectures/motion-algebras/angular-velocity/index) and the [exponential](/aiml-common/lectures/motion-algebras/exponential-coordinates/index) form this sequence:

$$
R(t) \xrightarrow{\;\text{differentiate}\;} \dot R \xrightarrow{\;R^{-1}\;} [\omega] \xrightarrow{\;\exp([\omega]\Delta t)\;} R(t + \Delta t) .
$$

Multiplication by $R^{-1}$ extracts the local motion. The exponential converts that motion into a finite rotation. In [exponential coordinates](/aiml-common/lectures/motion-algebras/exponential-coordinates/index#the-exponential-map-on-a-curved-space), an animation of a tangent plane on a sphere depicts the same wrapping process for the sphere $S^2$. For poses, the corresponding quantities are $T$, the twist $[\mathcal{V}_b]$, and $e^{[\mathcal{V}_b]\Delta t}$.

Consider the TurtleBot pose from [poses in SE(2)](#poses-in-se2).

1. **Velocity.** The robot travels forward at speed $v$ and turns at rate $\omega$. Its wheels cannot slide sideways, so its velocity in the body frame is the twist $\mathcal{V}_b = (\omega, v, 0)$. The sideways component is zero. In matrix form, the twist is
   $$
   [\mathcal{V}_b] = \begin{bmatrix} 0 & -\omega & v \\ \omega & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix} \in \mathfrak{se}(2) .
   $$
   This matrix belongs to the Lie algebra. In the notation of [Lie I](/aiml-common/lectures/motion-algebras/lie-theory/index), $[\mathcal{V}_b]$ is $\tau^\wedge$.
2. **Motion.** During a short time interval $\Delta t$, update the pose with
   $$
   T_{k+1} = T_k \exp\big([\mathcal{V}_b]\,\Delta t\big) .
   $$

When the twist stays constant, the same update is exact for an interval of any length. Write $\phi = \omega\,\Delta t$ for the angle turned. With no sideways speed, the exponential has the closed form

$$
\exp\big([\mathcal{V}_b]\,\Delta t\big) = \begin{bmatrix} \cos\phi & -\sin\phi & \dfrac{v}{\omega}\sin\phi \\ \sin\phi & \cos\phi & \dfrac{v}{\omega}(1 - \cos\phi) \\ 0 & 0 & 1 \end{bmatrix} , \qquad \omega \neq 0 .
$$

The rotation block turns the heading by $\phi$. The last column is the displacement in the body frame at the start of the interval: the robot travels along a circle of radius $v/\omega$. The center of that circle, the point $(0, v/\omega)$ in the starting body frame, is the fixed point $\mathbf{s}$ of a [planar screw motion](#planar-screw-motion). Solà, Deray and Atchuthan derive this formula for any twist in $\mathfrak{se}(2)$ [@Sola2018-ve].

For a numerical example, start the robot at $(x, y) = (1, 2)$, facing the $+y$ direction, so $\theta_0 = \pi/2$:

$$
T_0 = \begin{bmatrix} 0 & -1 & 1 \\ 1 & 0 & 2 \\ 0 & 0 & 1 \end{bmatrix} .
$$

It drives for $\Delta t = 1$ s at $v = 1$ m/s while turning left at $\omega = \pi/2$ rad/s. Then $\phi = \pi/2$, the radius is $2/\pi \approx 0.637$ m, and

$$
\exp\big([\mathcal{V}_b]\,\Delta t\big) = \begin{bmatrix} 0 & -1 & 2/\pi \\ 1 & 0 & 2/\pi \\ 0 & 0 & 1 \end{bmatrix} .
$$

In its own frame, the robot ends up $0.637$ m ahead and $0.637$ m to the left, turned by a quarter circle. Multiply on the right of $T_0$ to place this motion in the world:

$$
T_1 = T_0 \exp\big([\mathcal{V}_b]\,\Delta t\big) = \begin{bmatrix} -1 & 0 & 1 - 2/\pi \\ 0 & -1 & 2 + 2/\pi \\ 0 & 0 & 1 \end{bmatrix} \approx \begin{bmatrix} -1 & 0 & 0.363 \\ 0 & -1 & 2.637 \\ 0 & 0 & 1 \end{bmatrix} .
$$

The robot ends at $(0.363, 2.637)$, facing the $-x$ direction, so $\theta_1 = \pi$. The new position is not $(1, 2)$ plus $(0.637, 0.637)$. The body-frame displacement is first rotated by $R(\theta_0)$, which is the term $R_1 p_2$ from the composition of two poses in [poses in SE(2)](#poses-in-se2). Ahead of the robot is $+y$ in the world and to its left is $-x$, so the robot moves $0.637$ m in $+y$ and $0.637$ m in $-x$.

The code below builds the two matrices and repeats the update.

In [ ]:
def pose(theta, x, y):
    """The homogeneous transformation matrix of the planar pose (theta, x, y)."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, x], [s, c, y], [0.0, 0.0, 1.0]])


def exp_twist(omega, v, dt):
    """exp([V_b] dt) for the body twist V_b = (omega, v, 0)."""
    if abs(omega) < 1e-12:
        return np.array([[1.0, 0.0, v * dt], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
    phi, r = omega * dt, v / omega
    c, s = np.cos(phi), np.sin(phi)
    return np.array([[c, -s, r * s], [s, c, r * (1 - c)], [0.0, 0.0, 1.0]])


T0 = pose(np.pi / 2, 1.0, 2.0)
T1 = T0 @ exp_twist(np.pi / 2, 1.0, 1.0)
print(T1.round(3) + 0.0)  # + 0.0 turns -0.0 into 0.0

The product updates the position and the heading together, and the result is always a valid pose. Two commands in sequence give $T_2 = T_0\, e^{[\mathcal{V}_1]\Delta t_1} e^{[\mathcal{V}_2]\Delta t_2}$. The order matters, because the product of two exponentials is in general not the exponential of the sum.

Parallel parking chains three such commands. The robot starts at the origin, facing $+x$. It drives an arc to the left, then an arc to the right, then straight back. Each command keeps the twist constant, so each one is a single exponential.

In [ ]:
# (omega, v, duration) for each command: arc left, arc right, straight back
rho, alpha = 0.5, np.pi / 3
commands = [(1.0, rho, alpha), (-1.0, rho, alpha), (0.0, -1.0, 2 * rho * np.sin(alpha))]

T = pose(0.0, 0.0, 0.0)
for omega, v, duration in commands:
    T = T @ exp_twist(omega, v, duration)

theta = np.arctan2(T[1, 0], T[0, 0])
print("end (theta, x, y):", np.round([theta, T[0, 2], T[1, 2]], 3) + 0.0)

The robot ends at $(x, y) = (0, 0.5)$ with its starting heading. It has moved half a meter to its left, the direction its wheels forbid at every instant. [Configuration space](/aiml-common/lectures/motion-algebras/configuration-space/index) draws the same path in the space of all poses.

In [ ]:
rr.init("parallel_parking")
view = rrb.Spatial3DView(origin="world", name="Parallel parking", line_grid=False,
                         eye_controls=rrb.EyeControls3D(position=[0.43, -1.3, 1.1], look_target=[0.43, 0.25, 0.0], eye_up=[0, 0, 1]))
rr.save("parallel-parking.rrd", default_blueprint=view)
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)

# the floor, a TurtleBot-sized body with two wheels, and the robot's frame axes
gx, gy = np.arange(-0.5, 1.51, 0.25), np.arange(-0.5, 1.01, 0.25)
floor = [[[x, gy[0], 0], [x, gy[-1], 0]] for x in gx] + [[[gx[0], y, 0], [gx[-1], y, 0]] for y in gy]
rr.log("world/floor", rr.LineStrips3D(floor, colors=[90, 90, 90], radii=0.003), static=True)
rr.log("world/robot/body", rr.Cylinders3D(lengths=[0.08], radii=[0.12], centers=[[0, 0, 0.04]], colors=[[60, 64, 75]]), static=True)
rr.log("world/robot/wheels", rr.Boxes3D(centers=[[0, 0.13, 0.035], [0, -0.13, 0.035]],
                                        half_sizes=[[0.035, 0.012, 0.035]] * 2, colors=[[200, 200, 200]]), static=True)
rr.log("world/robot", rr.TransformAxes3D(axis_length=0.25), static=True)

# sample each command at 40 instants with the same exponential, so every pose is exact
names = ["arc left", "arc right", "straight back"]
t, T, trail = 0.0, pose(0.0, 0.0, 0.0), []
for name, (omega, v, duration) in zip(names, commands):
    for tau in np.linspace(0, duration, 40):
        Tk = T @ exp_twist(omega, v, tau)
        trail.append([Tk[0, 2], Tk[1, 2], 0.002])
        rr.set_time("time", duration=t + tau)
        rr.log("world/robot", rr.Transform3D(translation=[Tk[0, 2], Tk[1, 2], 0.0],
               rotation_axis_angle=rr.RotationAxisAngle(axis=[0, 0, 1], radians=np.arctan2(Tk[1, 0], Tk[0, 0]))))
        rr.log("world/path", rr.LineStrips3D([trail], colors=[43, 182, 115], radii=0.008))
        rr.log("world/command", rr.TextDocument(f"{name}: v = {v:+.2f} m/s, omega = {omega:+.2f} rad/s"))
    T, t = T @ exp_twist(omega, v, duration), t + duration

<iframe
  src="https://artifacts.aegeanai.com/rerun/viewer/0.41.0/index.html?url=https%3A%2F%2Fartifacts.aegeanai.com%2Frerun%2Fmotion-representations%2Fparallel-parking-0.41.rrd"
  width="100%"
  height="600px"
  style={{ border: "none", borderRadius: 8 }}
  allow="fullscreen"
  loading="lazy"
/>

*The parallel-parking path in the Rerun viewer. Drag the time slider to move the robot along the path, and drag in the 3D view to orbit it. The robot's frame moves with it: the red axis points forward and the green axis to its left. The path never runs along the green axis.*

[Wheeled robots](/aiml-common/lectures/kinematics/wheeled-robots/index) derives $(v, \omega)$ from the wheel speeds and applies this update to odometry. The chassis can point in any direction and reach any position. At each instant, the constraint $v_{by} = 0$ allows it to move only forward, backward, or turn.

<Note>
The TurtleBot runs ROS 2, which stores the same quantities in a different form.

- **Pose.** A [`geometry_msgs/Pose`](https://github.com/ros2/common_interfaces/blob/rolling/geometry_msgs/msg/Pose.msg) contains a position and an orientation [quaternion](/aiml-common/lectures/motion-algebras/quaternions/index) instead of a matrix $T$. A planar heading $\theta$ becomes the unit quaternion $(x, y, z, w) = (0, 0, \sin\tfrac{\theta}{2}, \cos\tfrac{\theta}{2})$. ROS lists the scalar part $w$ last.

- **Twist.** A [`geometry_msgs/Twist`](https://github.com/ros2/common_interfaces/blob/rolling/geometry_msgs/msg/Twist.msg) lists the linear velocity before the angular velocity, which reverses the Lynch and Park order. A drive command sets `linear.x` to $v$ and `angular.z` to $\omega$ and leaves the other four components at zero.

- **Odometry.** A [`nav_msgs/Odometry`](https://github.com/ros2/common_interfaces/blob/rolling/nav_msgs/msg/Odometry.msg) message gives the pose in a fixed frame and the twist in the robot's frame. Its twist is therefore the body twist $\mathcal{V}_b$.

- **Frames.** [REP 103](https://www.ros.org/reps/rep-0103.html) makes every frame right handed, with $x$ forward, $y$ left, and $z$ up on the robot.
</Note>

## Motion in space

### Poses in SE(3)

In three dimensions, the rotation $R$ is a $3 \times 3$ matrix in $SO(3)$ and the position $\mathbf{p}$ has three coordinates. The pose is a $4 \times 4$ matrix:

$$
T = \begin{bmatrix} R & \mathbf{p} \\ 0 & 1 \end{bmatrix} \in SE(3) .
$$

Everything in [poses in SE(2)](#poses-in-se2) carries over unchanged. The matrix acts on homogeneous coordinates in the same way, the inverse has the same form, and composition again produces the term $R_1 p_2$. As a manifold, SE(3) looks like $\mathbb{R}^3 \times SO(3)$, a position paired with an orientation. Its group is the semidirect product $SE(3) = \mathbb{R}^3 \rtimes SO(3)$.

### Twists

In three dimensions, the angular velocity is a vector $\omega \in \mathbb{R}^3$. A twist combines it with the linear velocity in one six-dimensional vector:

$$
\mathcal{V} = \begin{bmatrix} \omega \\ v \end{bmatrix} \in \mathbb{R}^6 , \qquad
[\mathcal{V}] = \begin{bmatrix} [\omega] & v \\ 0 & 0 \end{bmatrix} \in \mathfrak{se}(3) .
$$

$\mathfrak{se}(3)$ is the Lie algebra of $SE(3)$. The body twist and the space twist come from the same products as in the plane, $[\mathcal{V}_b] = T^{-1}\dot{T}$ and $[\mathcal{V}_s] = \dot{T} T^{-1}$. Their linear parts have the same meaning:

- $v_b = R^\top \dot{\mathbf{p}}$ is the velocity of the body-frame origin, written in the coordinates of $\{b\}$.

- $v_s = \dot{\mathbf{p}} - \omega_s \times \mathbf{p}$ is the velocity of the point of the body currently at the origin of $\{s\}$, written in $\{s\}$.

Unlike in the plane, the angular parts now differ as well: $\omega_s = R\,\omega_b$, because the two frames write the same rotation axis in different coordinates. The adjoint converts between the two twists:

$$
[\mathrm{Ad}_T] = \begin{bmatrix} R & 0 \\ [\mathbf{p}]R & R \end{bmatrix} , \qquad \mathcal{V}_s = [\mathrm{Ad}_T]\,\mathcal{V}_b .
$$

[Calculus on Lie groups](/aiml-common/lectures/motion-algebras/lie-calculus/index#where-the-modern-robotics-formulas-come-from) derives this matrix.

<Note>
Spatial-velocity conventions differ across systems, so integration may require conversion. [Warp](https://nvidia.github.io/warp/stable/language_reference/builtins.html#spatial-math) orders a spatial vector as (angular, linear). [PhysX](https://nvidia-omniverse.github.io/PhysX/physx/5.6.0/docs/Articulations.html#joint-positions-and-velocities) reports the linear velocity of the center of mass and the angular velocity, both in the world frame. The [Newton twist conventions](https://newton-physics.github.io/newton/stable/concepts/conventions.html#twist-conventions) compare these and other engines side by side.
</Note>

### Screw motion

[Exponential coordinates](/aiml-common/lectures/motion-algebras/exponential-coordinates/index) extend from rotations to poses. Let $\mathcal{S} = (\omega, v) \in \mathbb{R}^6$ be a screw axis. The displacement by $\theta$ along it is

$$
e^{[\mathcal{S}]\theta} = \begin{bmatrix} e^{[\omega]\theta} & G(\theta)v \\ 0 & 1 \end{bmatrix} ,
\qquad
G(\theta) = I\theta + (1 - \cos\theta)[\omega] + (\theta - \sin\theta)[\omega]^2 .
$$

Here $G(\theta)$ is Lynch and Park's notation. It is unrelated to the generator $G$ of Lie I. Given a transformation $T = (R, \mathbf{p})$, the matrix logarithm reverses this map. First, $\log R$ gives $[\omega]\theta$. Then $v = G(\theta)^{-1} \mathbf{p}$. Thus $[\mathcal{S}]\theta = \log T$.

Every pose can be reached by one screw motion, a rotation about an axis combined with a translation along it. This is the three-dimensional form of the [planar screw motion](#planar-screw-motion), where the axis points out of the page and the translation along it is zero. A body with a constant twist moves along a screw, and the update $T_{k+1} = T_k \exp\big([\mathcal{V}_b]\,\Delta t\big)$ from [velocity to motion](#from-velocity-to-motion) holds unchanged with $4 \times 4$ matrices.